In [2]:
# Load necessary csv files
import pandas as pd
import numpy as np
import re

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score
)


# PATHS
INPUT_FILE = Path(
    r"C:\Users\Visakh H S\Labour_Market_Intelligence\data\processed\master_jobs_industry.csv"
)

OUTPUT_FILE = Path(
    r"C:\Users\Visakh H S\Labour_Market_Intelligence\data\processed\master_jobs_industry_improved.csv"
)


# LOAD DATA
df = pd.read_csv(INPUT_FILE)

print("=" * 60)
print("DATASET LOADED")
print("=" * 60)

print("Rows:", len(df))
print("Columns:", df.columns.tolist())


# REQUIRED COLUMN CHECK
required_columns = [
    "job_id",
    "title",
    "description_clean",
    "industry"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

print("\nRequired columns found successfully.")

display(
    df[
        [
            "job_id",
            "title",
            "description_clean",
            "industry"
        ]
    ].head()
)

DATASET LOADED
Rows: 19207
Columns: ['job_id', 'source_id', 'title', 'company', 'location', 'description', 'salary_min', 'salary_max', 'salary_avg', 'created', 'contract_type', 'contract_time', 'url', 'source', 'title_clean', 'description_clean', 'industry']

Required columns found successfully.


,job_id,title,description_clean,industry
0,0d6b3088-b9b3-4238-8be0-9ef2704750ac,Dispenser NVQ,join superdrug pharmacy team whether qualified...,Healthcare
1,58cb2c2b-6f47-4ffa-aec2-dde531278d24,Service Colleague Express,job title service colleague express location e...,Retail
2,ba4c3cc1-dde8-4122-9f64-5edfd3297e11,Service Colleague Express,job title service colleague express location e...,Retail
3,f8ec9f92-d35a-4058-b376-8b43222d0e50,Customer Delivery Driver,job title customer delivery driver location st...,Transport and Logistics
4,4b4bfd85-4575-409f-8f48-25a3bb4cdc7b,IT Security Technical Delivery Manager VP,leading banking client looking talented motiva...,IT and Technology


In [3]:
# Prepare classification text

df["title"] = df["title"].fillna("").astype(str)
df["description_clean"] = df["description_clean"].fillna("").astype(str)

df["classification_text"] = (
    df["title"] + " " +
    df["title"] + " " +
    df["title"] + " " +
    df["description_clean"]
)

df["classification_text"] = (
    df["classification_text"]
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

print(df["classification_text"].head(3))

0    dispenser nvq dispenser nvq dispenser nvq join...
1    service colleague express service colleague ex...
2    service colleague express service colleague ex...
Name: classification_text, dtype: object


In [4]:
# Current industry distribution
print("Current industry distribution:")

industry_counts = df["industry"].value_counts()

display(industry_counts)

Current industry distribution:


industry
Other                            13651
Engineering                       1043
Education                          912
Finance and Banking                664
Manufacturing                      596
Construction                       564
Transport and Logistics            381
IT and Technology                  370
Hospitality and Food Services      249
Human Resources                    155
Healthcare                         142
Legal                              136
Automotive                         124
Retail                             113
Marketing and Advertising           63
Real Estate                         22
Insurance                           22
Name: count, dtype: int64

In [5]:
# INDUSTRY_RULES
INDUSTRY_RULES = {

    "IT and Technology": [
        r"\bsoftware engineer\b",
        r"\bsoftware developer\b",
        r"\bweb developer\b",
        r"\bfull stack\b",
        r"\bfrontend developer\b",
        r"\bbackend developer\b",
        r"\bdata scientist\b",
        r"\bdata analyst\b",
        r"\bmachine learning\b",
        r"\bai engineer\b",
        r"\bartificial intelligence\b",
        r"\bpython developer\b",
        r"\bjava developer\b",
        r"\bc\# developer\b",
        r"\bdevops\b",
        r"\bcyber ?security\b",
        r"\binformation security\b",
        r"\bcloud engineer\b",
        r"\bcloud architect\b",
        r"\bsoftware architect\b",
        r"\bsystems analyst\b",
        r"\bit support\b",
        r"\bit technician\b",
        r"\bnetwork engineer\b",
        r"\bnetwork administrator\b",
        r"\bdatabase administrator\b",
        r"\bdata engineer\b",
        r"\btechnical architect\b",
    ],

    "Healthcare": [
        r"\bpharmacist\b",
        r"\bpharmacy\b",
        r"\bpharmacy assistant\b",
        r"\bpharmacy colleague\b",
        r"\bdispenser\b",
        r"\bnurse\b",
        r"\bnursing\b",
        r"\bcare assistant\b",
        r"\bhealthcare assistant\b",
        r"\bhealth care assistant\b",
        r"\bdoctor\b",
        r"\bphysician\b",
        r"\bclinical\b",
        r"\bmedical\b",
        r"\bdentist\b",
        r"\bdental\b",
        r"\btherapist\b",
        r"\bradiographer\b",
    ],

    "Finance and Banking": [
        r"\baccountant\b",
        r"\baccounting\b",
        r"\baudit\b",
        r"\bauditor\b",
        r"\bfinancial analyst\b",
        r"\bfinance analyst\b",
        r"\bfinance manager\b",
        r"\bfinancial controller\b",
        r"\bpayroll\b",
        r"\binvestment analyst\b",
        r"\binvestment banker\b",
        r"\binvestment banking\b",
        r"\bprivate equity\b",
        r"\basset management\b",
        r"\bfund manager\b",
        r"\bportfolio manager\b",
        r"\brisk analyst\b",
        r"\brisk manager\b",
        r"\bmortgage\b",
        r"\bbanking\b",
        r"\btreasury\b",
    ],

    "Construction": [
        r"\belectrician\b",
        r"\bplumber\b",
        r"\bbricklayer\b",
        r"\bcarpenter\b",
        r"\bjoiner\b",
        r"\bwelder\b",
        r"\bconstruction\b",
        r"\bsite manager\b",
        r"\bsite supervisor\b",
        r"\bquantity surveyor\b",
        r"\bbuilding surveyor\b",
        r"\bconstruction manager\b",
        r"\bbuilding services\b",
    ],

    "Engineering": [
        r"\bmechanical engineer\b",
        r"\belectrical engineer\b",
        r"\bcivil engineer\b",
        r"\bdesign engineer\b",
        r"\bprocess engineer\b",
        r"\bmanufacturing engineer\b",
        r"\bengineering manager\b",
        r"\bengineering technician\b",
        r"\bengineer\b",
        r"\bengineering\b",
    ],

    "Education": [
        r"\bteacher\b",
        r"\bteaching assistant\b",
        r"\blecturer\b",
        r"\btutor\b",
        r"\bprofessor\b",
        r"\beducation\b",
        r"\bteacher assistant\b",
        r"\bschool\b",
        r"\bcollege\b",
    ],

    "Retail": [
        r"\bsales assistant\b",
        r"\bstore manager\b",
        r"\bstore assistant\b",
        r"\bretail assistant\b",
        r"\bretail manager\b",
        r"\bshop assistant\b",
        r"\bcustomer assistant\b",
        r"\bservice colleague\b",
        r"\bsales colleague\b",
        r"\bshop manager\b",
        r"\bretail\b",
    ],

    "Transport and Logistics": [
        r"\bdelivery driver\b",
        r"\bcustomer delivery driver\b",
        r"\bhgv driver\b",
        r"\bvan driver\b",
        r"\btruck driver\b",
        r"\blorry driver\b",
        r"\bwarehouse\b",
        r"\bwarehouse operative\b",
        r"\bwarehouse manager\b",
        r"\blogistics\b",
        r"\btransport\b",
        r"\bforklift\b",
        r"\breach truck\b",
    ],

    "Automotive": [
        r"\bvehicle technician\b",
        r"\bautomotive\b",
        r"\bcar technician\b",
        r"\bvehicle mechanic\b",
        r"\bmotor mechanic\b",
        r"\bvehicle sales\b",
        r"\bcar sales\b",
        r"\bmechanic\b",
    ],

    "Hospitality and Food Services": [
        r"\bchef\b",
        r"\bcook\b",
        r"\brestaurant\b",
        r"\bwaiter\b",
        r"\bwaitress\b",
        r"\bbarista\b",
        r"\bcatering\b",
        r"\bhospitality\b",
        r"\bfood services\b",
        r"\brestaurant manager\b",
        r"\bhotel\b",
    ],

    "Human Resources": [
        r"\bhuman resources\b",
        r"\bhr manager\b",
        r"\bhr advisor\b",
        r"\bhr administrator\b",
        r"\brecruitment\b",
        r"\brecruiter\b",
        r"\btalent acquisition\b",
    ],

    "Legal": [
        r"\bsolicitor\b",
        r"\bbarrister\b",
        r"\bparalegal\b",
        r"\blegal advisor\b",
        r"\blegal counsel\b",
        r"\blegal assistant\b",
        r"\blawyer\b",
        r"\blegal\b",
    ],

    "Marketing and Advertising": [
        r"\bmarketing\b",
        r"\bdigital marketing\b",
        r"\bmarketing manager\b",
        r"\bmarketing executive\b",
        r"\bseo\b",
        r"\bcontent marketing\b",
        r"\badvertising\b",
        r"\bsocial media manager\b",
    ],

    "Real Estate": [
        r"\breal estate\b",
        r"\bproperty manager\b",
        r"\bproperty management\b",
        r"\bestate agent\b",
        r"\bproperty consultant\b",
        r"\blettings\b",
    ],

    "Insurance": [
        r"\binsurance\b",
        r"\bunderwriter\b",
        r"\bunderwriting\b",
        r"\binsurance broker\b",
        r"\binsurance advisor\b",
        r"\bclaims handler\b",
        r"\bclaims adjuster\b",
    ],

    "Manufacturing": [
        r"\bmanufacturing\b",
        r"\bproduction operative\b",
        r"\bproduction manager\b",
        r"\bproduction supervisor\b",
        r"\bassembly operative\b",
        r"\bmachine operator\b",
        r"\bmanufacturing operative\b",
    ]
}

In [6]:
# rule_classify()
def rule_classify(title):

    title = str(title).lower()

    matches = []

    for industry, patterns in INDUSTRY_RULES.items():

        for pattern in patterns:

            if re.search(pattern, title):

                matches.append(industry)
                break

    # Only return a rule when exactly one industry matches
    if len(matches) == 1:
        return matches[0]

    # Multiple conflicting rules → let ML decide
    return None


df["industry_rule"] = df["title"].apply(rule_classify)

print("Rule-based classifications:")
print(df["industry_rule"].notna().sum())

print("\nUnclassified by rules:")
print(df["industry_rule"].isna().sum())

print("\nRule distribution:")
display(
    df["industry_rule"]
    .value_counts()
)

Rule-based classifications:
7806

Unclassified by rules:
11401

Rule distribution:


industry_rule
Engineering                      2499
Finance and Banking               927
Manufacturing                     805
Education                         788
Construction                      632
Transport and Logistics           628
Hospitality and Food Services     287
Legal                             210
Automotive                        207
Healthcare                        179
Human Resources                   176
Retail                            153
IT and Technology                 135
Marketing and Advertising         118
Insurance                          34
Real Estate                        28
Name: count, dtype: int64

In [7]:
# inspect rule classifications
display(
    df[
        df["industry_rule"].notna()
    ][
        [
            "title",
            "industry",
            "industry_rule"
        ]
    ].sample(
        min(50, df["industry_rule"].notna().sum()),
        random_state=42
    )
)

,title,industry,industry_rule
11351,Financial Controller,Finance and Banking,Finance and Banking
17868,Become a Secondary school Tutor in Ovingdean –...,Education,Education
14433,Payroll & People Administrator,Other,Finance and Banking
1441,Electrical Engineer,Engineering,Engineering
11414,Chief Electronics Engineer,Other,Engineering
19098,Maths Teacher – Excellent School in Redditch –...,Education,Education
3030,Senior Geotechnical / Geoenvironmental Engineer,Other,Engineering
1933,Retail Optical Assistant,Other,Retail
11799,Manufacturing Team Leader,Other,Manufacturing
15266,CAD Engineer,Other,Engineering


In [8]:
# create rule training data
rule_training = df[
    df["industry_rule"].notna()
].copy()

rule_training["training_industry"] = (
    rule_training["industry_rule"]
)

print(
    "High-confidence rule training jobs:",
    len(rule_training)
)

print("\nTraining distribution:")

display(
    rule_training["training_industry"]
    .value_counts()
)

High-confidence rule training jobs: 7806

Training distribution:


training_industry
Engineering                      2499
Finance and Banking               927
Manufacturing                     805
Education                         788
Construction                      632
Transport and Logistics           628
Hospitality and Food Services     287
Legal                             210
Automotive                        207
Healthcare                        179
Human Resources                   176
Retail                            153
IT and Technology                 135
Marketing and Advertising         118
Insurance                          34
Real Estate                        28
Name: count, dtype: int64

In [9]:
# existing labelled data
existing_training = df[
    (df["industry"] != "Other") &
    (df["industry"].notna()) &
    (df["industry"].astype(str).str.strip() != "")
].copy()

existing_training["training_industry"] = (
    existing_training["industry"]
)

print(
    "Existing labelled training jobs:",
    len(existing_training)
)

Existing labelled training jobs: 5556


In [10]:
# combine training data
training_df = pd.concat(
    [
        existing_training[
            [
                "classification_text",
                "training_industry"
            ]
        ],

        rule_training[
            [
                "classification_text",
                "training_industry"
            ]
        ]
    ],
    ignore_index=True
)

# Remove duplicate training examples
training_df = training_df.drop_duplicates(
    subset=["classification_text", "training_industry"]
)

print(
    "Final training examples:",
    len(training_df)
)

print("\nTraining labels:")
display(
    training_df["training_industry"]
    .value_counts()
)

Final training examples: 7903

Training labels:


training_industry
Engineering                      2397
Finance and Banking               916
Education                         868
Manufacturing                     797
Construction                      653
Transport and Logistics           549
IT and Technology                 379
Hospitality and Food Services     281
Legal                             213
Healthcare                        207
Human Resources                   180
Automotive                        145
Retail                            127
Marketing and Advertising         117
Insurance                          39
Real Estate                        35
Name: count, dtype: int64

In [11]:
# TF-IDF
X_text = training_df["classification_text"]
y = training_df["training_industry"]

X_train, X_test, y_train, y_test = train_test_split(
    X_text,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    max_features=100000
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Training matrix:", X_train_tfidf.shape)
print("Testing matrix:", X_test_tfidf.shape)

Training matrix: (6322, 38359)
Testing matrix: (1581, 38359)


In [12]:
# Logistic Regression
classifier = LogisticRegression(
    max_iter=2000,
    class_weight="balanced"
)

classifier.fit(
    X_train_tfidf,
    y_train
)

print("Industry classifier trained successfully.")

Industry classifier trained successfully.


In [13]:
# validation
y_pred = classifier.predict(X_test_tfidf)

accuracy = accuracy_score(
    y_test,
    y_pred
)

macro_f1 = f1_score(
    y_test,
    y_pred,
    average="macro"
)

weighted_f1 = f1_score(
    y_test,
    y_pred,
    average="weighted"
)

print("INDUSTRY CLASSIFIER VALIDATION")

print("Accuracy:", round(accuracy, 4))
print("Macro F1:", round(macro_f1, 4))
print("Weighted F1:", round(weighted_f1, 4))

print("\nClassification Report:")

print(
    classification_report(
        y_test,
        y_pred,
        zero_division=0
    )
)

INDUSTRY CLASSIFIER VALIDATION
Accuracy: 0.9652
Macro F1: 0.9545
Weighted F1: 0.9656

Classification Report:
                               precision    recall  f1-score   support

                   Automotive       1.00      0.93      0.96        29
                 Construction       0.98      0.97      0.98       131
                    Education       0.99      0.97      0.98       174
                  Engineering       0.99      0.96      0.97       480
          Finance and Banking       0.98      0.98      0.98       183
                   Healthcare       0.89      0.95      0.92        41
Hospitality and Food Services       0.98      0.98      0.98        56
              Human Resources       0.92      1.00      0.96        36
            IT and Technology       0.81      0.97      0.89        76
                    Insurance       1.00      1.00      1.00         8
                        Legal       0.93      0.98      0.95        43
                Manufacturing       0.

In [14]:
# classify all 19,207 jobs
all_tfidf = vectorizer.transform(
    df["classification_text"]
)

ml_predictions = classifier.predict(
    all_tfidf
)

ml_probabilities = classifier.predict_proba(
    all_tfidf
)

ml_confidence = ml_probabilities.max(axis=1)

df["industry_ml"] = ml_predictions
df["industry_ml_confidence"] = ml_confidence

print("ML classification completed.")

ML classification completed.


In [15]:
# hybrid classification
def hybrid_industry(row):

    rule = row["industry_rule"]
    ml = row["industry_ml"]
    confidence = row["industry_ml_confidence"]
    existing = row["industry"]

    
# 1. High-confidence title rule
    if pd.notna(rule):
        return rule


# 2. High-confidence ML prediction
    if confidence >= 0.65:
        return ml


# 3. Keep existing reliable classification
    if (
        pd.notna(existing)
        and str(existing).strip() != ""
        and existing != "Other"
    ):
        return existing

# 4. Otherwise retain Other
    return "Other"


df["industry_improved"] = df.apply(
    hybrid_industry,
    axis=1
)

print("Hybrid classification completed.")

Hybrid classification completed.


In [16]:
# comparison
comparison = pd.DataFrame({
    "Original": df["industry"].value_counts(),
    "Improved": df["industry_improved"].value_counts()
}).fillna(0).astype(int)

display(comparison)

,Original,Improved
Automotive,124,239
Construction,564,1167
Education,912,996
Engineering,1043,2561
Finance and Banking,664,972
Healthcare,142,228
Hospitality and Food Services,249,301
Human Resources,155,200
IT and Technology,370,420
Insurance,22,47


In [17]:
# original and Improved version
print("Original Other:")
print(
    (df["industry"] == "Other").sum()
)

print("\nImproved Other:")
print(
    (df["industry_improved"] == "Other").sum()
)

Original Other:
13651

Improved Other:
9818


In [18]:
# changed classifications
changed = df[
    df["industry"] != df["industry_improved"]
].copy()

print(
    "Number of changed classifications:",
    len(changed)
)

display(
    changed[
        [
            "title",
            "industry",
            "industry_rule",
            "industry_ml",
            "industry_ml_confidence",
            "industry_improved"
        ]
    ].head(100)
)

Number of changed classifications: 3860


,title,industry,industry_rule,industry_ml,industry_ml_confidence,industry_improved
56,EE Retail Guide,Other,Retail,Retail,0.411493,Retail
60,FLT / Counterbalance,Other,None,Transport and Logistics,0.867706,Transport and Logistics
61,No Experience Required - Trainee Sales & Marke...,Other,Marketing and Advertising,Marketing and Advertising,0.553310,Marketing and Advertising
63,Cleaning Operative - Primary School,Other,Education,Education,0.521640,Education
68,Recruitment Resourcer,Other,Human Resources,Human Resources,0.540591,Human Resources
...,...,...,...,...,...,...
489,Head of Transport Solutions Development,Other,Transport and Logistics,Transport and Logistics,0.334155,Transport and Logistics
498,Send Ta,Other,None,Education,0.759617,Education
506,Digital Forensics R&D Engineer,Other,Engineering,Engineering,0.481846,Engineering
509,HR Administrator,Other,Human Resources,Human Resources,0.814580,Human Resources


In [19]:
# final distribution
print("FINAL INDUSTRY DISTRIBUTION")

final_distribution = (
    df["industry_improved"]
    .value_counts()
)

display(final_distribution)

print("\nPercentages:")

display(
    (
        df["industry_improved"]
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
        .to_frame("percentage")
    )
)

FINAL INDUSTRY DISTRIBUTION


industry_improved
Other                            9818
Engineering                      2561
Construction                     1167
Education                         996
Finance and Banking               972
Manufacturing                     907
Transport and Logistics           780
IT and Technology                 420
Hospitality and Food Services     301
Automotive                        239
Healthcare                        228
Legal                             227
Human Resources                   200
Retail                            185
Marketing and Advertising         123
Insurance                          47
Real Estate                        36
Name: count, dtype: int64


Percentages:


,percentage
industry_improved,
Other,51.12
Engineering,13.33
Construction,6.08
Education,5.19
Finance and Banking,5.06
Manufacturing,4.72
Transport and Logistics,4.06
IT and Technology,2.19
Hospitality and Food Services,1.57


In [20]:
# save final dataset
# Replace the old industry column with the improved one
df["industry_original"] = df["industry"]

df["industry"] = df["industry_improved"]

# Remove temporary classification columns
df = df.drop(
    columns=[
        "classification_text",
        "industry_rule",
        "industry_ml",
        "industry_ml_confidence",
        "industry_improved"
    ],
    errors="ignore"
)

df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("DATASET SAVED")
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("File:", OUTPUT_FILE)

DATASET SAVED
Rows: 19207
Columns: 18
File: C:\Users\Visakh H S\Labour_Market_Intelligence\data\processed\master_jobs_industry_improved.csv


#### QUALITY CHECK AND SECOND PASS CLASSIFIER

In [21]:
# IMPROVED INDUSTRY QUALITY VALIDATION

print("=" * 70)
print("IMPROVED INDUSTRY QUALITY VALIDATION")
print("=" * 70)

# Check final dataset
print("\nRows:", len(df))
print("Columns:", len(df.columns))

# 1. Check missing industries
missing_industry = (
    df["industry"].isna() |
    (df["industry"].astype(str).str.strip() == "")
)

print("\nMissing industries:", missing_industry.sum())


# 2. Check industry values
valid_industries = sorted(INDUSTRY_RULES.keys())

print("\nValid industry categories:")
for industry in valid_industries:
    print("-", industry)


# 3. Find unexpected industry labels
unexpected = df[
    ~df["industry"].isin(valid_industries + ["Other"])
]["industry"].value_counts()

print("\nUnexpected industry labels:")
print(unexpected)


# 4. Final distribution
print("\nFinal industry distribution:")

final_counts = df["industry"].value_counts()

display(final_counts)


# 5. Final percentages
print("\nFinal industry percentages:")

final_percentages = (
    df["industry"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .to_frame("percentage")
)

display(final_percentages)


# 6. Duplicate job IDs
print("\nDuplicate job IDs:", df["job_id"].duplicated().sum())


# 7. Duplicate complete rows
# Avoid the previous unhashable-list problem
# by checking duplicates only on scalar columns.

duplicate_columns = [
    col for col in [
        "job_id",
        "title",
        "company",
        "location",
        "created"
    ]
    if col in df.columns
]

print(
    "Duplicate jobs based on identifying fields:",
    df.duplicated(subset=duplicate_columns).sum()
)

print("\n" + "=" * 70)
print("VALIDATION COMPLETE")
print("=" * 70)

IMPROVED INDUSTRY QUALITY VALIDATION

Rows: 19207
Columns: 18

Missing industries: 0

Valid industry categories:
- Automotive
- Construction
- Education
- Engineering
- Finance and Banking
- Healthcare
- Hospitality and Food Services
- Human Resources
- IT and Technology
- Insurance
- Legal
- Manufacturing
- Marketing and Advertising
- Real Estate
- Retail
- Transport and Logistics

Unexpected industry labels:
Series([], Name: count, dtype: int64)

Final industry distribution:


industry
Other                            9818
Engineering                      2561
Construction                     1167
Education                         996
Finance and Banking               972
Manufacturing                     907
Transport and Logistics           780
IT and Technology                 420
Hospitality and Food Services     301
Automotive                        239
Healthcare                        228
Legal                             227
Human Resources                   200
Retail                            185
Marketing and Advertising         123
Insurance                          47
Real Estate                        36
Name: count, dtype: int64


Final industry percentages:


,percentage
industry,
Other,51.12
Engineering,13.33
Construction,6.08
Education,5.19
Finance and Banking,5.06
Manufacturing,4.72
Transport and Logistics,4.06
IT and Technology,2.19
Hospitality and Food Services,1.57



Duplicate job IDs: 0
Duplicate jobs based on identifying fields: 0

VALIDATION COMPLETE


In [22]:
# INVESTIGATE "OTHER" CLASSIFICATIONS
print("=" * 70)
print("INVESTIGATING OTHER INDUSTRY CLASSIFICATIONS")
print("=" * 70)

other_df = df[
    df["industry"].astype(str).str.strip().str.lower() == "other"
].copy()

print("\nJobs classified as Other:", len(other_df))
print(
    "Percentage of dataset:",
    round(len(other_df) / len(df) * 100, 2),
    "%"
)


# Show random sample
sample_size = min(100, len(other_df))

other_sample = other_df.sample(
    n=sample_size,
    random_state=42
)

display(
    other_sample[
        [
            "job_id",
            "title",
            "description_clean"
        ]
    ].reset_index(drop=True)
)

INVESTIGATING OTHER INDUSTRY CLASSIFICATIONS

Jobs classified as Other: 9818
Percentage of dataset: 51.12 %


,job_id,title,description_clean
0,a15354dc-d243-4cf7-8e60-213b5badf59d,Moulding Technician,leading manufacturer plastic product looking r...
1,e9e849b1-91be-480d-8beb-9c6ccdf82654,Operations Manager,job profile operation manager rf operation man...
2,f5e119f9-244a-466e-831d-1dfa423ebec0,Side Loader,position end line loader location holywell hou...
3,1b2d0acf-d8ee-4528-88a9-bf96a45df0c3,Associate Project Director - Nuclear,mace combine construction expertise consultanc...
4,c016c391-e931-4ff9-abaf-541f57e81e1b,Quality Inspector,elevation recruitment group delighted working ...
...,...,...,...
95,a8418348-9b9d-4534-835d-57ffe14b30ea,Client Onboarding Specialist,client onboarding specialist permanent hybrid ...
96,341fb157-1416-46c9-8e40-b9cdaa0ec1f8,Immediate Start Accounts Payable Clerkq,client manufacturing business based telford cu...
97,14b31965-ba5d-4303-82c1-bf7466e22dbc,Class 1 Driver,job title class driver location hatfield pay r...
98,e201f59b-c76d-47ca-9b35-d8054560841f,Receptionist,join team receptionist location wilmslow contr...


In [23]:
# SECOND-PASS INDUSTRY CLASSIFICATION FOR "OTHER"
print("=" * 70)
print("SECOND-PASS CLASSIFICATION OF OTHER JOBS")
print("=" * 70)


# Industry-specific TITLE keywords
SECOND_PASS_RULES = {

    "Manufacturing": [
        "manufacturing",
        "manufacturing technician",
        "production operative",
        "production technician",
        "production manager",
        "production supervisor",
        "machine operator",
        "machine technician",
        "moulding technician",
        "molding technician",
        "quality inspector",
        "quality technician",
        "assembly operative",
        "assembly technician",
        "process technician",
        "factory operative",
        "factory worker",
        "fabricator",
        "welder"
    ],

    "Transport and Logistics": [
        "class 1 driver",
        "class 2 driver",
        "hgv driver",
        "lgv driver",
        "lorry driver",
        "truck driver",
        "delivery driver",
        "van driver",
        "warehouse operative",
        "warehouse worker",
        "warehouse manager",
        "forklift driver",
        "forklift operator",
        "reach truck",
        "counterbalance",
        "loader",
        "side loader",
        "logistics coordinator",
        "logistics manager",
        "transport manager",
        "transport coordinator"
    ],

    "Finance and Banking": [
        "accounts payable",
        "accounts receivable",
        "accountant",
        "accounting assistant",
        "finance assistant",
        "finance manager",
        "financial analyst",
        "financial controller",
        "payroll administrator",
        "payroll officer",
        "bookkeeper",
        "credit controller",
        "credit analyst",
        "investment analyst",
        "private equity",
        "banking",
        "banker"
    ],

    "IT and Technology": [
        "software developer",
        "software engineer",
        "data analyst",
        "data scientist",
        "machine learning engineer",
        "ai engineer",
        "developer",
        "programmer",
        "devops",
        "cloud engineer",
        "systems administrator",
        "system administrator",
        "network engineer",
        "cybersecurity",
        "cyber security",
        "information security",
        "security analyst",
        "it support",
        "it technician",
        "database administrator"
    ],

    "Engineering": [
        "mechanical engineer",
        "electrical engineer",
        "civil engineer",
        "design engineer",
        "project engineer",
        "engineering manager",
        "engineering technician",
        "process engineer",
        "maintenance engineer",
        "structural engineer",
        "systems engineer",
        "chemical engineer",
        "nuclear engineer",
        "engineering consultant"
    ],

    "Construction": [
        "site manager",
        "site supervisor",
        "site engineer",
        "quantity surveyor",
        "construction manager",
        "construction worker",
        "bricklayer",
        "carpenter",
        "plumber",
        "electrician",
        "roofer",
        "groundworker",
        "building surveyor"
    ],

    "Healthcare": [
        "nurse",
        "nursing",
        "doctor",
        "pharmacist",
        "pharmacy",
        "healthcare assistant",
        "care assistant",
        "clinical",
        "healthcare",
        "medical",
        "therapist",
        "radiographer",
        "paramedic",
        "dentist",
        "dental"
    ],

    "Education": [
        "teacher",
        "teaching assistant",
        "lecturer",
        "professor",
        "tutor",
        "education",
        "school administrator",
        "academic",
        "head teacher",
        "teacher assistant"
    ],

    "Legal": [
        "solicitor",
        "barrister",
        "paralegal",
        "legal assistant",
        "legal secretary",
        "lawyer",
        "legal counsel",
        "compliance lawyer"
    ],

    "Human Resources": [
        "hr manager",
        "hr advisor",
        "hr adviser",
        "hr assistant",
        "human resources",
        "recruitment consultant",
        "recruiter",
        "talent acquisition",
        "people manager",
        "people advisor"
    ],

    "Marketing and Advertising": [
        "marketing manager",
        "marketing assistant",
        "marketing executive",
        "digital marketing",
        "social media manager",
        "content manager",
        "content creator",
        "seo specialist",
        "seo manager",
        "advertising manager",
        "brand manager"
    ],

    "Hospitality and Food Services": [
        "restaurant manager",
        "restaurant general manager",
        "chef",
        "head chef",
        "kitchen manager",
        "waiter",
        "waitress",
        "bartender",
        "bar manager",
        "hotel manager",
        "hotel receptionist",
        "hospitality",
        "catering"
    ],

    "Retail": [
        "sales assistant",
        "retail assistant",
        "retail manager",
        "store manager",
        "store assistant",
        "shop assistant",
        "shop manager",
        "retail supervisor",
        "cashier",
        "customer service assistant"
    ],

    "Automotive": [
        "automotive technician",
        "vehicle technician",
        "motor vehicle technician",
        "vehicle mechanic",
        "car mechanic",
        "automotive engineer",
        "motor mechanic",
        "vehicle mechanic"
    ],

    "Real Estate": [
        "estate agent",
        "property manager",
        "property administrator",
        "property consultant",
        "real estate",
        "letting agent",
        "lettings negotiator"
    ],

    "Insurance": [
        "insurance advisor",
        "insurance adviser",
        "insurance broker",
        "insurance underwriter",
        "insurance analyst",
        "claims handler",
        "claims advisor"
    ]
}



# Second-pass classifier
def second_pass_classify(title):
    
    title = str(title).lower().strip()

    if not title:
        return None

    matches = []

    for industry, keywords in SECOND_PASS_RULES.items():

        for keyword in keywords:

            keyword_pattern = r"\b" + re.escape(keyword.lower()) + r"\b"

            if re.search(keyword_pattern, title):
                matches.append(industry)
                break

    # Only classify if there is exactly one industry match
    if len(matches) == 1:
        return matches[0]

    # Ambiguous → leave unchanged
    return None


# Apply only to current "Other" jobs
other_mask = (
    df["industry"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("other")
)

df["second_pass_industry"] = None

df.loc[other_mask, "second_pass_industry"] = (
    df.loc[other_mask, "title"]
    .apply(second_pass_classify)
)


# Count results
second_pass_matches = (
    df["second_pass_industry"]
    .notna()
)

print(
    "\nSecond-pass classifications:",
    second_pass_matches.sum()
)

print(
    "Remaining Other:",
    other_mask.sum() - second_pass_matches.sum()
)

print("\nSecond-pass industry distribution:")

display(
    df.loc[
        second_pass_matches,
        "second_pass_industry"
    ].value_counts()
)

SECOND-PASS CLASSIFICATION OF OTHER JOBS

Second-pass classifications: 707
Remaining Other: 9111

Second-pass industry distribution:


second_pass_industry
Manufacturing                    281
IT and Technology                190
Transport and Logistics           94
Finance and Banking               92
Hospitality and Food Services     15
Marketing and Advertising         14
Construction                       7
Retail                             4
Human Resources                    4
Education                          2
Healthcare                         2
Insurance                          1
Real Estate                        1
Name: count, dtype: int64

In [24]:
# INSPECT SECOND-PASS CLASSIFICATIONS
second_pass_df = df[
    df["second_pass_industry"].notna()
].copy()

print(
    "Total second-pass classifications:",
    len(second_pass_df)
)

display(
    second_pass_df[
        [
            "job_id",
            "title",
            "industry",
            "second_pass_industry",
            "description_clean"
        ]
    ]
    .sample(
        n=min(100, len(second_pass_df)),
        random_state=42
    )
    .reset_index(drop=True)
)

Total second-pass classifications: 707


,job_id,title,industry,second_pass_industry,description_clean
0,7a8587ee-0a57-406f-bcd9-351b9fae7b6b,Roofer - Teams,Other,Construction,roofer roofer mate roofing team cornwall resou...
1,dcf293ef-937e-44b0-8622-54ebe09fafc1,AI Engineer,Other,IT and Technology,klipboard introduced flexible hybrid work poli...
2,16f8b214-c612-4e4a-8f4b-6fdf7d57ed40,Factory Operative,Other,Manufacturing,factory operative per hour plus overtime day s...
3,065781b5-b29b-43f0-be09-d3ab7bd26782,Senior Quantitative Developer - Risk & Pricing...,Other,IT and Technology,compensation competitive financial service tra...
4,e7460729-0a3a-4ac4-88cf-8ac416ddde6a,Senior .Net Developer (Billing or Financial Sy...,Other,IT and Technology,senior net developer billing financial system ...
...,...,...,...,...,...
95,78b3e20f-bd5a-4824-9604-37741fa4d94d,HGV Class 2 Driver,Other,Transport and Logistics,class shuttle driver wellesbourne cv long term...
96,ee6bc849-7050-46fe-b5f1-00064cdbc42d,Credit Controller,Other,Finance and Banking,marc daniel urgently seeking credit controller...
97,d0f4f41b-4c60-4300-b4d5-c3d4a668787b,Finance Assistant – Central London – Hybrid - ...,Other,Finance and Banking,finance assistant central london hybrid highly...
98,5e59f0c5-fbfc-43df-9d54-2d01a995994e,Fabricator,Other,Manufacturing,looking hand detail oriented fabricator join g...


In [25]:
# APPLY SECOND-PASS CLASSIFICATIONS
before_other = (
    df["industry"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("other")
    .sum()
)

# Keep the original industry for auditability
df["industry_before_second_pass"] = df["industry"]

# Replace Other only where the second-pass classifier
# produced a confident single-industry match
second_pass_mask = df["second_pass_industry"].notna()

df.loc[
    second_pass_mask,
    "industry"
] = df.loc[
    second_pass_mask,
    "second_pass_industry"
]

after_other = (
    df["industry"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("other")
    .sum()
)

print("=" * 70)
print("SECOND-PASS CLASSIFICATION APPLIED")
print("=" * 70)

print("\nClassifications applied:", second_pass_mask.sum())
print("Other before:", before_other)
print("Other after:", after_other)

print(
    "\nOther reduction:",
    before_other - after_other
)

print(
    "Other percentage after:",
    round(after_other / len(df) * 100, 2),
    "%"
)

SECOND-PASS CLASSIFICATION APPLIED

Classifications applied: 707
Other before: 9818
Other after: 9111

Other reduction: 707
Other percentage after: 47.44 %


In [26]:
# UPDATED INDUSTRY DISTRIBUTION
print("=" * 70)
print("UPDATED INDUSTRY DISTRIBUTION")
print("=" * 70)

industry_counts = df["industry"].value_counts()

display(industry_counts)

print("\nPercentages:")

industry_percentages = (
    df["industry"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
    .to_frame("percentage")
)

display(industry_percentages)

UPDATED INDUSTRY DISTRIBUTION


industry
Other                            9111
Engineering                      2561
Manufacturing                    1188
Construction                     1174
Finance and Banking              1064
Education                         998
Transport and Logistics           874
IT and Technology                 610
Hospitality and Food Services     316
Automotive                        239
Healthcare                        230
Legal                             227
Human Resources                   204
Retail                            189
Marketing and Advertising         137
Insurance                          48
Real Estate                        37
Name: count, dtype: int64


Percentages:


,percentage
industry,
Other,47.44
Engineering,13.33
Manufacturing,6.19
Construction,6.11
Finance and Banking,5.54
Education,5.20
Transport and Logistics,4.55
IT and Technology,3.18
Hospitality and Food Services,1.65


In [27]:
df["industry_before_second_pass"] = df["industry"]

second_pass_mask = df["second_pass_industry"].notna()

df.loc[second_pass_mask, "industry"] = (
    df.loc[second_pass_mask, "second_pass_industry"]
)

In [28]:
print("Classifications applied:", second_pass_mask.sum())
print("Remaining Other:", (df["industry"] == "Other").sum())

Classifications applied: 707
Remaining Other: 9111


In [29]:
# save
INDUSTRY_OUTPUT = r"C:\Users\Visakh H S\Labour_Market_Intelligence\data\processed\master_jobs_industry_improved.csv"

df.to_csv(
    INDUSTRY_OUTPUT,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", INDUSTRY_OUTPUT)
print("Rows:", len(df))

Saved: C:\Users\Visakh H S\Labour_Market_Intelligence\data\processed\master_jobs_industry_improved.csv
Rows: 19207
